# Lab 2 | ACF/PACF + đề xuất tham số

**Mục bình chọn:** ACF/PACF + đề xuất tham số — cả 2 dataset

**Hạn bàn giao:** 24/10/2026, 20:00 — bàn giao cấu hình ứng viên cho ARIMA/SARIMA.

> Bạn đọc ACF/PACF để đề xuất các cấu hình đáng thử, kèm lý do. Không chỉ vẽ hai hình rồi chọn số ngẫu nhiên, và chưa cần tự fit toàn bộ mô hình.

**Đối chiếu đề:** Bài cơ bản: Bài 4 trên Dataset 1. Nâng cao: ACF/PACF, tham số thường p/q và mùa vụ P/Q của Dataset 2.

Đây là mô tả nhận việc, không phải lời giải. Các cell code vẫn giữ nguyên `# TODO`.

## Đầu vào, phụ thuộc và việc làm song song

**Cần nhận từ ADF:** code/phép xử lý để tái tạo chuỗi train sau xử lý, d/D đề xuất và kết quả kiểm định.
**Có thể chuẩn bị trước:** import, khung vẽ, cách chọn số lag và phần trình bày.
**Cần chờ để chốt:** kết quả xử lý chuỗi; không kết luận tham số từ test hoặc từ chuỗi chưa phù hợp.

## Cụ thể cần làm gì?

1. Tái tạo chuỗi train theo đúng phép xử lý được bàn giao; xác nhận đang dùng đúng dataset và index.
2. Vẽ ACF/PACF cho cả hai dataset với số lag phù hợp; hình phải có nhãn và khoảng tin cậy.
3. Nhận xét dạng cắt cụt/suy giảm và các lag nổi bật; Dataset 2 chú ý các lag mùa vụ 12, 24, ...
4. Dataset 1: đề xuất 1–2 bộ (p,d,q), trong đó d đối chiếu với phần ADF; viết lý do chọn từng bộ.
5. Dataset 2: đề xuất cấu hình ARIMA baseline và cấu hình SARIMA (p,d,q)(P,D,Q,12).
   Phân biệt tham số thường và mùa vụ, không tự đổi d/D mà chưa trao đổi.
6. Trả lời câu hỏi Bài 4: ACF/PACF thể hiện điều gì, cắt cụt/suy giảm ở đâu và gợi ý mô hình nào.

## Cần bàn giao những gì?

**Gửi cho phần ARIMA và SARIMA:**
- Các đồ thị ACF/PACF có nhận xét, chỉ rõ chuỗi đã xử lý nào được dùng để vẽ.
- 1–2 cấu hình ARIMA Dataset 1; cấu hình ARIMA baseline Dataset 2; cấu hình SARIMA Dataset 2.
- Mỗi cấu hình ghi đủ tham số, dataset, lý do và điều còn chưa chắc chắn.
- Đây là ứng viên để fit/kiểm tra tiếp; ACF/PACF không tự chứng minh mô hình dự báo tốt.

## Quy ước chung

- Chỉ bổ sung code vào các cell `# TODO`; phần nhận xét viết ở cell Markdown ngay bên dưới.
- Đọc CSV chung trong `data/`; không sửa nguồn, không tạo một bản dataset khác.
- Dataset 1 (`a10.csv`) dùng `date`/`value`; Dataset 2 (`airline-passengers.csv`) dùng `Month`/`Passengers`.
- Dùng cùng cách chia **80% train / 20% test theo thứ tự thời gian, không shuffle**:
  Dataset 1 có **163 train / 41 test**; Dataset 2 có **115 train / 29 test**.
- Có thể dùng helper `src/lab2_data.py`, nhưng phần import/load vẫn để bạn tự viết.
  Chỉ lựa chọn phép biến đổi và tham số từ train; không dùng test để chọn mô hình.
- Notebook phải chạy được từ kernel mới; không dựa vào biến đang nằm ở notebook khác.
- Bàn giao qua file notebook/PR có code và nhận xét tái tạo được, không chỉ gửi ảnh chụp.

## Khi nào xem là hoàn thành?

- [x] Phân tích cả hai dataset và dùng đúng chuỗi train sau xử lý.
- [x] Có cấu hình thường, mùa vụ và lý do chứ không chỉ có hình.
- [x] Người fit mô hình nhận được tham số đủ rõ để chạy ngay.

---
## Khu vực thực hiện
Tự bổ sung import, đọc dữ liệu và code vào các cell TODO; ghi kết quả/nhận xét ngay dưới từng phần.

### 1. Tiếp nhận và tái tạo chuỗi đã xử lý

Nhận phép xử lý/d/D từ phần ADF; phân tích chuỗi train đã xử lý.

In [ ]:
import numpy as np
import pandas as pd
from src.lab2_data import load_datasets, chronological_split, SEASON_LENGTH

# Load dữ liệu chuẩn từ module src.lab2_data của dự án
_, _, y1, y2 = load_datasets()
train1, test1 = chronological_split(y1)
train2, test2 = chronological_split(y2)

# Tái tạo chuỗi sai phân bậc 1 (d=1) cho Dataset 1 theo Bài 3 ADF
train1_d1 = train1.diff(1).dropna()

# Hàm tái tạo chuỗi đã xử lý cả sai phân thường và mùa vụ (d=1, D=1, m=12)
def make_stationary(series, d, D, m=SEASON_LENGTH):
    out = series.copy()
    for _ in range(d):
        out = out.diff(1)
    for _ in range(D):
        out = out.diff(m)
    return out.dropna()

# Chuỗi train sau sai phân thường và sai phân mùa vụ
stat1 = make_stationary(train1, d=1, D=1, m=SEASON_LENGTH)
stat2 = make_stationary(train2, d=1, D=1, m=SEASON_LENGTH)

print("Dataset 1 - Train gốc:", train1.shape, "sau d=1:", train1_d1.shape, "sau d=1, D=1:", stat1.shape)
print("Dataset 2 - Train gốc:", train2.shape, "sau d=1, D=1:", stat2.shape)

**Nhận xét:**
- **Dataset 1 (a10):** Đã tái tạo chuỗi train sau sai phân bậc 1 ($d=1$, 162 quan sát) theo kết luận Bài 3 kiểm định ADF (chuỗi đã đạt tính dừng ở mức ý nghĩa 5%). Đồng thời, chuẩn bị thêm chuỗi sau cả sai phân thường và mùa vụ ($d=1, D=1$, 150 quan sát) để đối chiếu đặc trưng mùa vụ.
- **Dataset 2 (AirPassengers):** Đã tái tạo chuỗi train sau sai phân bậc 1 và sai phân mùa vụ ($d=1, D=1, m=12$, 102 quan sát) theo bàn giao từ bước ADF để khử cả xu hướng lẫn tính mùa vụ.
- **Nguyên tắc dữ liệu:** Tất cả các phép biến đổi sai phân chỉ áp dụng trên tập huấn luyện (`train1`, `train2`), tuyệt đối không dùng tập test nhằm ngăn ngừa rò rỉ dữ liệu (data leakage).

### 2. Vẽ và diễn giải ACF/PACF

Vẽ ACF/PACF cả hai dataset; giải thích p/q/P/Q.

In [ ]:
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# 1. Vẽ ACF và PACF cho Dataset 1 (sau sai phân bậc 1 d=1 phục vụ mô hình ARIMA Bài 4 & 5)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(train1_d1, ax=axes[0], lags=24, title="Dataset 1 (a10) - ACF (d=1)")
plot_pacf(train1_d1, ax=axes[1], lags=24, title="Dataset 1 (a10) - PACF (d=1)")
plt.tight_layout()
plt.show()

# 2. Vẽ ACF và PACF cho Dataset 1 (sau sai phân d=1, D=1 để đối chiếu thành phần phi mùa vụ)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(stat1, ax=axes[0], lags=24, title="Dataset 1 (a10) - ACF (d=1, D=1)")
plot_pacf(stat1, ax=axes[1], lags=24, title="Dataset 1 (a10) - PACF (d=1, D=1)")
plt.tight_layout()
plt.show()

# 3. Vẽ ACF và PACF cho Dataset 2 (sau sai phân d=1, D=1, s=12 phục vụ SARIMA)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(stat2, ax=axes[0], lags=36, title="Dataset 2 (AirPassengers) - ACF (d=1, D=1)")
plot_pacf(stat2, ax=axes[1], lags=36, title="Dataset 2 (AirPassengers) - PACF (d=1, D=1)")
plt.tight_layout()
plt.show()

**Nhận xét và Trả lời câu hỏi Bài 4:**

1. **Dataset 1 (a10 - chuỗi sau $d=1$):**
   - **ACF:** Tự tương quan giảm dần nhưng xuất hiện đỉnh nhọn vượt ngưỡng tin cậy rõ rệt tại lag mùa vụ $12$ và $24$, cho thấy chuỗi vẫn mang tính mùa vụ mạnh theo chu kỳ năm.
   - **PACF:** Cắt cụt rõ rệt sau lag 1 (hoặc lag 2), các lag sau đó nằm trong dải tin cậy ngoại trừ các lag bội số của 12.
   - **Đối chiếu trên chuỗi đã khử mùa vụ ($d=1, D=1$):** PACF cắt cụt ở lag 1 (hoặc 2), ACF cắt cụt ở lag 1. Điều này củng cố cấu hình thành phần phi mùa vụ phù hợp là $p \in \{1, 2\}$ và $q=1$.

2. **Dataset 2 (AirPassengers - chuỗi sau $d=1, D=1$):**
   - **Thành phần phi mùa vụ ($p, q$):** 
     - Biểu đồ ACF cắt cụt sau lag 1 ($q=1$).
     - Biểu đồ PACF giảm dần hoặc cắt cụt sau lag 1 ($p=1$ hoặc $p=0$).
   - **Thành phần mùa vụ ($P, Q$ tại các lag bội số của $s=12$):**
     - Trên biểu đồ ACF: Có một điểm âm rõ rệt vượt ngưỡng tin cậy tại lag 12 và cắt cụt ở lag 24, 36 $\rightarrow$ Gợi ý mạnh mẽ thành phần Moving Average mùa vụ bậc 1 ($Q=1$).
     - Trên biểu đồ PACF: Các giá trị tại lag mùa vụ 12, 24 suy giảm dần $\rightarrow$ Gợi ý thành phần Autoregressive mùa vụ bậc 0 ($P=0$).

3. **Trả lời trực tiếp câu hỏi Bài 4 (PDF):**
   - *ACF cắt cụt ở lag nào?* Ở phần phi mùa vụ, ACF cắt cụt rõ ở lag 1 ($q=1$).
   - *PACF cắt cụt ở lag nào?* PACF cắt cụt rõ sau lag 1 hoặc 2 ($p=1$ hoặc $p=2$).
   - *Gợi ý mô hình ARIMA(p,d,q)?* Gợi ý cấu hình chính là **ARIMA(1,1,1)** và cấu hình đối chiếu là **ARIMA(2,1,1)**.

### 3. Đề xuất cấu hình và bàn giao

Đề xuất cấu hình ARIMA và SARIMA, không dùng test để chọn.

In [ ]:
proposals = {
    "Dataset_1_ARIMA": [
        {"model": "ARIMA(1,1,1)", "p": 1, "d": 1, "q": 1, "note": "Ứng viên chính: PACF cắt cụt lag 1, ACF cắt cụt lag 1"},
        {"model": "ARIMA(2,1,1)", "p": 2, "d": 1, "q": 1, "note": "Ứng viên dự phòng: kiểm tra thêm autoregressive bậc 2"}
    ],
    "Dataset_2_ARIMA_Baseline": [
        {"model": "ARIMA(1,1,1)", "p": 1, "d": 1, "q": 1, "note": "ARIMA baseline đối chiếu với SARIMA"},
        {"model": "ARIMA(2,1,1)", "p": 2, "d": 1, "q": 1, "note": "ARIMA baseline bậc cao hơn"}
    ],
    "Dataset_2_SARIMA": [
        {"model": "SARIMA(0,1,1)(0,1,1)_12", "p": 0, "d": 1, "q": 1, "P": 0, "D": 1, "Q": 1, "s": 12, "note": "Airline Model kinh điển của Box-Jenkins (ACF spike âm tại lag 12)"},
        {"model": "SARIMA(1,1,1)(0,1,1)_12", "p": 1, "d": 1, "q": 1, "P": 0, "D": 1, "Q": 1, "s": 12, "note": "Mở rộng thêm thành phần phi mùa vụ AR(1)"}
    ]
}

import pprint
pprint.pprint(proposals)

**Nhận xét và Bàn giao cấu hình:**
- **Bàn giao cho phần ARIMA (Dataset 1 & Dataset 2 Baseline):**
  - Dataset 1: Cấu hình chính là `ARIMA(1,1,1)` và cấu hình phụ `ARIMA(2,1,1)` fit trên `train1` với $d=1$.
  - Dataset 2: Cấu hình baseline `ARIMA(1,1,1)` fit trên `train2` với $d=1$ để làm mốc đối chứng đánh giá hiệu quả của thành phần mùa vụ.
- **Bàn giao cho phần SARIMA (Dataset 2):**
  - Đề xuất cấu hình chuẩn Box-Jenkins Airline Model: `SARIMA(0,1,1)(0,1,1)_12` ($d=1, D=1, s=12$) dựa trên đặc trưng ACF cắt cụt lag 1 và spike âm tại lag 12.
  - Cấu hình thử nghiệm bổ sung: `SARIMA(1,1,1)(0,1,1)_12` để kiểm tra đóng góp của $p=1$.